# Unpack Firebase Data — UIST Multi-Turn Study

This notebook unpacks the Firebase JSON export into three dataframes:
1. **Participant Data**: One row per participant with conditions, surveys, MBA/MBE trait ratings, attention checks
2. **Conversation Data**: One row per message across both sessions
3. **Persona Vector Data**: One row per persona snapshot from per-session `personaVectorLog`

**Study**: `multiturn-pilot2` — Two-session within-subjects design
- Session 1: Baseline (no visualization for any condition)
- Session 2: Experimental (visualization condition applied)
- 3 conditions: control (0), single-turn (1), multi-turn (2)
- Prompt counterbalancing: ASST/ROLEPLY order randomized
- 6 traits on -10 to +10 scale: empathy, erudite, robotic, romantic, sycophantic, toxic

In [1]:
import json
import pandas as pd
import numpy as np
from datetime import datetime

## 1. Load JSON Data

In [2]:
# Load and merge JSON exports (pilot1 excluded — missing s1 MBA and incomplete survey)
json_paths = [
    '/Users/sheerkarny/Downloads/mech-chat-ee0c5-default-rtdb-multiturn-pilot2-export (2).json',
    '/Users/sheerkarny/Downloads/mech-chat-ee0c5-default-rtdb-multiturn-run3-export (7).json',
    '/Users/sheerkarny/Downloads/mech-chat-ee0c5-default-rtdb-multiturn-run4-export (4).json',
]

participant_data = {}
for json_path in json_paths:
    with open(json_path, 'r') as f:
        raw = f.read()
    idx = raw.index('{')
    data = json.loads(raw[idx:])

    if 'participantData' in data:
        pd_chunk = data['participantData']
    else:
        for key in data:
            if isinstance(data[key], dict) and 'participantData' in data[key]:
                pd_chunk = data[key]['participantData']
                break
        else:
            raise KeyError(f"Could not find 'participantData' in {json_path}")

    overlap = set(pd_chunk.keys()) & set(participant_data.keys())
    if overlap:
        print(f"  {len(overlap)} duplicate firebase IDs — keeping latest")
    participant_data.update(pd_chunk)
    print(f"Loaded {len(pd_chunk)} from {json_path.split('/')[-1]}")

print(f"\nTotal unique participants: {len(participant_data)}")

Loaded 36 from mech-chat-ee0c5-default-rtdb-multiturn-pilot2-export (2).json
Loaded 98 from mech-chat-ee0c5-default-rtdb-multiturn-run3-export (7).json
Loaded 130 from mech-chat-ee0c5-default-rtdb-multiturn-run4-export (4).json

Total unique participants: 264


## 2. Create Participant-Level Dataframe

In [3]:
TRAITS = ['empathy', 'erudite', 'robotic', 'romantic', 'sycophantic', 'toxic']

# UEQ item keys as they appear in the export
UEQ_ITEMS = [
    'obstructive_supportive',
    'complicated_easy',
    'inefficient_efficient',
    'confusing_clear',
    'boring_exciting',
    'not_interesting_interesting',
    'conventional_inventive',
    'usual_leading_edge',
]


def extract_participant_info(firebase_id, p_data):
    """Extract all relevant participant information for statistical analysis."""
    row = {'firebase_id': firebase_id}

    # --- Identifiers (from urlParameters) ---
    url_params = p_data.get('urlParameters', {})
    row['prolific_id'] = url_params.get('PROLIFIC_PID', None)
    row['session_id'] = url_params.get('SESSION_ID', None)
    row['study_id'] = url_params.get('STUDY_ID', None)

    # --- Experiment Condition ---
    exp = p_data.get('experimentCondition', {})
    row['visualization_condition'] = exp.get('visualizationCondition', None)
    row['condition_name'] = exp.get('conditionName', None)
    row['assignment_method'] = exp.get('assignmentMethod', None)
    row['prompt_order'] = exp.get('promptOrder', None)
    row['session1_prompt_type'] = exp.get('session1PromptType', None)
    row['session2_prompt_type'] = exp.get('session2PromptType', None)
    row['condition_timestamp'] = exp.get('timestamp', None)

    # --- Pre-Survey (before any sessions) ---
    pre = p_data.get('preSurvey', {})
    row['pre_predictability'] = pre.get('predictability', None)
    row['pre_negative_predictability'] = pre.get('negativePredictability', None)
    row['pre_trust'] = pre.get('trust', None)
    row['pre_survey_timestamp'] = pre.get('timestamp', None)

    # --- Per-Session Data ---
    for session_num in [1, 2]:
        s_key = f'session{session_num}'
        s_prefix = f's{session_num}'
        session = p_data.get(s_key, {})

        # Prompt reading
        prompt_reading = session.get('promptReading', {})
        row[f'{s_prefix}_prompt_type'] = prompt_reading.get('promptType', None)
        row[f'{s_prefix}_prompt_shown'] = prompt_reading.get('systemPromptShown', None)

        # System prompt text
        sys_prompt = session.get('systemPrompt', {})
        row[f'{s_prefix}_system_prompt'] = sys_prompt.get('prompt', None)

        # Attention checks
        attn = session.get('attentionChecks', {})

        # Prompt recall: "Is the AI instructed to always obey the user?"
        prompt_recall = attn.get('promptRecall', {})
        row[f'{s_prefix}_attention_prompt_recall_answer'] = prompt_recall.get('answer', None)
        row[f'{s_prefix}_attention_prompt_recall_correct'] = prompt_recall.get('correctAnswer', None)
        row[f'{s_prefix}_attention_prompt_recall_passed'] = prompt_recall.get('passed', None)

        # Trait scale MBA: select +3 attention check
        trait_mba_attn = attn.get('traitScaleMBA', {})
        row[f'{s_prefix}_attention_mba_answer'] = trait_mba_attn.get('answer', None)
        row[f'{s_prefix}_attention_mba_passed'] = trait_mba_attn.get('passed', None)

        # MBA (Model Behavior Anticipation — before chat)
        mba = session.get('mba', {})
        mba_predictions = mba.get('traitPredictions', {})
        for trait in TRAITS:
            row[f'{s_prefix}_mba_{trait}'] = mba_predictions.get(trait, None)
        row[f'{s_prefix}_mba_timestamp'] = mba.get('timestamp', None)

        # MBE (Model Behavior Evaluation — after chat)
        mbe = session.get('mbe', {})
        mbe_predictions = mbe.get('traitPredictions', {})
        for trait in TRAITS:
            row[f'{s_prefix}_mbe_{trait}'] = mbe_predictions.get(trait, None)
        row[f'{s_prefix}_mbe_timestamp'] = mbe.get('timestamp', None)

        # Chat timer (directly under session, not chat)
        timer = session.get('timer', {})
        row[f'{s_prefix}_chat_start'] = timer.get('startTime', None)
        row[f'{s_prefix}_chat_end'] = timer.get('endTime', None)
        row[f'{s_prefix}_chat_duration'] = timer.get('duration', None)

        # Chat message counts (messages is a list directly under session)
        messages = session.get('messages', [])
        if isinstance(messages, list):
            msg_list = [m for m in messages if m is not None]
        elif isinstance(messages, dict):
            msg_list = [v for v in messages.values() if v is not None]
        else:
            msg_list = []
        row[f'{s_prefix}_num_user_messages'] = len([m for m in msg_list if m.get('role') == 'user'])
        row[f'{s_prefix}_num_assistant_messages'] = len([m for m in msg_list if m.get('role') == 'assistant'])
        row[f'{s_prefix}_num_total_messages'] = len(msg_list)

    # --- Final Survey (task4FinalSurvey) ---
    final = p_data.get('task4FinalSurvey', {})
    row['final_survey_timestamp'] = final.get('timestamp', None)
    row['final_survey_condition'] = final.get('condition', None)

    # Likert items
    likert = final.get('likert', {})
    row['final_predictability_post'] = likert.get('predictabilityPost', None)
    row['final_negative_predictability_post'] = likert.get('negativePredictabilityPost', None)
    row['final_trust_post'] = likert.get('trustPost', None)
    # Visualization-specific (conditions 1-2 only)
    row['final_viz_helpfulness'] = likert.get('visualizationHelpfulness', None)
    row['final_viz_anticipation'] = likert.get('visualizationAnticipation', None)
    row['final_viz_prediction'] = likert.get('visualizationPrediction', None)
    row['final_viz_referenced'] = likert.get('visualizationReferenced', None)
    row['final_viz_confidence'] = likert.get('visualizationConfidence', None)
    row['final_viz_frequency'] = likert.get('visualizationFrequency', None)
    row['final_viz_comprehension'] = likert.get('visualizationComprehension', None)
    # Condition 2 only
    row['final_drift_panel_helpful'] = likert.get('driftPanelHelpful', None)

    # UEQ (conditions 1-2 only) — keys are descriptive like 'boring_exciting'
    ueq = final.get('ueq', {})
    for item_key in UEQ_ITEMS:
        row[f'final_ueq_{item_key}'] = ueq.get(item_key, None)

    # Open-ended
    open_ended = final.get('openEnded', {})
    row['final_interaction_reflection'] = open_ended.get('interactionReflection', None)
    row['final_visualization_usage'] = open_ended.get('visualizationUsage', None)
    row['final_visualization_feedback'] = open_ended.get('visualizationFeedback', None)
    row['final_general_feedback'] = open_ended.get('generalFeedback', None)

    return row


# Build dataframe
participant_rows = []
for firebase_id, p_data in participant_data.items():
    participant_rows.append(extract_participant_info(firebase_id, p_data))

df_participants = pd.DataFrame(participant_rows)

print(f"Participant dataframe shape: {df_participants.shape}")
print(f"\nColumns ({len(df_participants.columns)}):")
for col in df_participants.columns:
    print(f"  {col}")

Participant dataframe shape: (264, 96)

Columns (96):
  firebase_id
  prolific_id
  session_id
  study_id
  visualization_condition
  condition_name
  assignment_method
  prompt_order
  session1_prompt_type
  session2_prompt_type
  condition_timestamp
  pre_predictability
  pre_negative_predictability
  pre_trust
  pre_survey_timestamp
  s1_prompt_type
  s1_prompt_shown
  s1_system_prompt
  s1_attention_prompt_recall_answer
  s1_attention_prompt_recall_correct
  s1_attention_prompt_recall_passed
  s1_attention_mba_answer
  s1_attention_mba_passed
  s1_mba_empathy
  s1_mba_erudite
  s1_mba_robotic
  s1_mba_romantic
  s1_mba_sycophantic
  s1_mba_toxic
  s1_mba_timestamp
  s1_mbe_empathy
  s1_mbe_erudite
  s1_mbe_robotic
  s1_mbe_romantic
  s1_mbe_sycophantic
  s1_mbe_toxic
  s1_mbe_timestamp
  s1_chat_start
  s1_chat_end
  s1_chat_duration
  s1_num_user_messages
  s1_num_assistant_messages
  s1_num_total_messages
  s2_prompt_type
  s2_prompt_shown
  s2_system_prompt
  s2_attention_prompt

## 3. Data Quality Filtering

In [4]:
print("=" * 60)
print("DATA QUALITY FILTERING")
print("=" * 60)

total_before = len(df_participants)
print(f"\nTotal participants before filtering: {total_before}")

# Check 1: Must have Prolific ID
missing_prolific = df_participants['prolific_id'].isna().sum()
print(f"Missing Prolific ID: {missing_prolific}")

# Check 2: Must have completed the final survey
missing_final = df_participants['final_survey_timestamp'].isna().sum()
print(f"Missing final survey timestamp: {missing_final}")

# Apply base filters
df_participants = df_participants[
    df_participants['prolific_id'].notna() &
    df_participants['final_survey_timestamp'].notna()
].copy()
print(f"After base filters (prolific + final survey): {len(df_participants)}")

# Check 2b: Must have chat messages in both sessions
missing_chat = df_participants[
    (df_participants['s1_num_total_messages'] == 0) |
    (df_participants['s2_num_total_messages'] == 0)
]
if len(missing_chat) > 0:
    print(f"Missing chat messages in at least one session: {len(missing_chat)}")
    for _, row in missing_chat.iterrows():
        print(f"  {row['prolific_id']}  s1={row['s1_num_total_messages']}  s2={row['s2_num_total_messages']}")
    df_participants = df_participants[
        (df_participants['s1_num_total_messages'] > 0) &
        (df_participants['s2_num_total_messages'] > 0)
    ].copy()
    print(f"After chat message filter: {len(df_participants)}")
else:
    print(f"All participants have chat messages in both sessions")

# Check 3: Validate survey completeness per condition
# Control: 3 base likert + 2 open-ended
# Single-turn: base + 7 viz likert + 8 UEQ + 2 viz open-ended
# Multi-turn: base + 7 viz likert + driftPanelHelpful + 8 UEQ + 2 viz open-ended

REQUIRED_LIKERT_ALL = ['final_predictability_post', 'final_negative_predictability_post', 'final_trust_post']
REQUIRED_LIKERT_VIZ = ['final_viz_helpfulness', 'final_viz_anticipation', 'final_viz_prediction',
                       'final_viz_referenced', 'final_viz_confidence', 'final_viz_frequency',
                       'final_viz_comprehension']
REQUIRED_LIKERT_MT = ['final_drift_panel_helpful']
REQUIRED_UEQ = [f'final_ueq_{k}' for k in [
    'obstructive_supportive', 'complicated_easy', 'inefficient_efficient', 'confusing_clear',
    'boring_exciting', 'not_interesting_interesting', 'conventional_inventive', 'usual_leading_edge']]
REQUIRED_OE_VIZ = ['final_visualization_usage', 'final_visualization_feedback']

def check_survey_complete(row):
    """Return True if participant has all required survey items for their condition."""
    cond = row['condition_name']

    # All conditions need base likert
    for col in REQUIRED_LIKERT_ALL:
        if pd.isna(row.get(col)):
            return False

    # Viz conditions (single_turn, multi_turn) need extra items
    if cond in ('single_turn', 'multi_turn'):
        for col in REQUIRED_LIKERT_VIZ + REQUIRED_UEQ + REQUIRED_OE_VIZ:
            if pd.isna(row.get(col)):
                return False

    # Multi-turn also needs drift panel
    if cond == 'multi_turn':
        for col in REQUIRED_LIKERT_MT:
            if pd.isna(row.get(col)):
                return False

    return True

df_participants['survey_complete'] = df_participants.apply(check_survey_complete, axis=1)
incomplete = df_participants[~df_participants['survey_complete']]
if len(incomplete) > 0:
    print(f"\nIncomplete final survey (missing condition-specific items): {len(incomplete)}")
    for _, row in incomplete.iterrows():
        print(f"  {row['prolific_id']}  cond={row['condition_name']}")

# --- Final Analysis Set ---
df_participants_final = df_participants[df_participants['survey_complete']].copy()
df_participants_final = df_participants_final.drop(columns=['survey_complete'])

print(f"\n--- FINAL ANALYSIS SET ---")
print(f"Total participants: {len(df_participants_final)}")

# --- Exploratory Set: completed both sessions (has s2 MBE) ---
df_all = pd.DataFrame(participant_rows)  # re-use the unfiltered df
df_participants_exploratory = df_all[
    df_all['prolific_id'].notna() &
    df_all['s2_mbe_timestamp'].notna()
].copy()
df_participants_exploratory['in_final_analysis'] = (
    df_participants_exploratory['firebase_id'].isin(df_participants_final['firebase_id'])
)

print(f"\n--- EXPLORATORY SET (completed both sessions) ---")
print(f"Total participants: {len(df_participants_exploratory)}")
print(f"  In final analysis: {df_participants_exploratory['in_final_analysis'].sum()}")
print(f"  Exploratory only: {(~df_participants_exploratory['in_final_analysis']).sum()}")

# Check for duplicate Prolific IDs
for name, df in [("Final", df_participants_final), ("Exploratory", df_participants_exploratory)]:
    dupes = df['prolific_id'].duplicated().sum()
    print(f"\nDuplicate Prolific IDs ({name}): {dupes}")
    if dupes > 0:
        print(df[df['prolific_id'].duplicated(keep=False)][['prolific_id', 'firebase_id']])

# Use final as primary
df_participants = df_participants_final

DATA QUALITY FILTERING

Total participants before filtering: 264
Missing Prolific ID: 1
Missing final survey timestamp: 35
After base filters (prolific + final survey): 229
Missing chat messages in at least one session: 1
  6709a18c842035499e0f1353  s1=0  s2=2
After chat message filter: 228

--- FINAL ANALYSIS SET ---
Total participants: 228

--- EXPLORATORY SET (completed both sessions) ---
Total participants: 239
  In final analysis: 228
  Exploratory only: 11

Duplicate Prolific IDs (Final): 0

Duplicate Prolific IDs (Exploratory): 0


In [5]:
for label, df in [("FINAL ANALYSIS", df_participants), ("EXPLORATORY", df_participants_exploratory)]:
    print("=" * 60)
    print(f"CONDITION COUNTS — {label}")
    print("=" * 60)

    print(f"\nTotal participants: {len(df)}")

    print(f"\n--- Visualization Condition ---")
    print(df['visualization_condition'].value_counts().sort_index())
    print(f"\n--- Condition Name ---")
    print(df['condition_name'].value_counts())

    print(f"\n--- Prompt Order ---")
    print(df['prompt_order'].value_counts())

    print(f"\n--- Condition x Prompt Order Crosstab ---")
    print(pd.crosstab(df['condition_name'], df['prompt_order']))

    print(f"\n--- Assignment Method ---")
    print(df['assignment_method'].value_counts())
    print()

CONDITION COUNTS — FINAL ANALYSIS

Total participants: 228

--- Visualization Condition ---
visualization_condition
0    63
1    85
2    80
Name: count, dtype: int64

--- Condition Name ---
condition_name
single_turn    85
multi_turn     80
control        63
Name: count, dtype: int64

--- Prompt Order ---
prompt_order
asst_first       116
roleply_first    112
Name: count, dtype: int64

--- Condition x Prompt Order Crosstab ---
prompt_order    asst_first  roleply_first
condition_name                           
control                 31             32
multi_turn              40             40
single_turn             45             40

--- Assignment Method ---
assignment_method
random    228
Name: count, dtype: int64

CONDITION COUNTS — EXPLORATORY

Total participants: 239

--- Visualization Condition ---
visualization_condition
0    73
1    85
2    81
Name: count, dtype: int64

--- Condition Name ---
condition_name
single_turn    85
multi_turn     81
control        73
Name: count, dtyp

## 4. Attention Check Summary

In [6]:
print("=" * 60)
print("ATTENTION CHECK RESULTS")
print("=" * 60)

attention_cols = []
for s in [1, 2]:
    for check_name, col_suffix in [('Prompt Recall', 'prompt_recall_passed'), ('MBA Trait Scale', 'mba_passed')]:
        col = f's{s}_attention_{col_suffix}'
        attention_cols.append(col)
        passed = df_participants[col].fillna(False).astype(bool).sum()
        total = df_participants[col].notna().sum()
        print(f"Session {s} — {check_name}: {passed}/{total} passed "
              f"({passed/max(total,1)*100:.1f}%)")

# Overall: did participant pass ALL attention checks?
df_participants['attention_all_passed'] = df_participants[attention_cols].apply(
    lambda row: all(row.fillna(False).astype(bool)), axis=1
)

print(f"\nPassed ALL attention checks: "
      f"{df_participants['attention_all_passed'].sum()}/{len(df_participants)}")
print(f"Failed at least one: {(~df_participants['attention_all_passed']).sum()}")

# Show which checks were failed
failed = df_participants[~df_participants['attention_all_passed']]
if len(failed) > 0:
    print(f"\nFailed participants detail:")
    for _, p in failed.iterrows():
        fails = [col for col in attention_cols
                 if not bool(p.get(col, False)) or pd.isna(p.get(col))]
        print(f"  {p['prolific_id']}: failed {fails}")

ATTENTION CHECK RESULTS
Session 1 — Prompt Recall: 117/228 passed (51.3%)
Session 1 — MBA Trait Scale: 219/228 passed (96.1%)
Session 2 — Prompt Recall: 112/228 passed (49.1%)
Session 2 — MBA Trait Scale: 223/228 passed (97.8%)

Passed ALL attention checks: 12/228
Failed at least one: 216

Failed participants detail:
  6637b21240220c2517fffa1a: failed ['s1_attention_prompt_recall_passed']
  672c7f537c4a19882d6f9e15: failed ['s1_attention_prompt_recall_passed', 's2_attention_prompt_recall_passed']
  69a6e19366f31b3e3460f2b7: failed ['s2_attention_prompt_recall_passed']
  66f0409cf2658a66284b3b2d: failed ['s2_attention_prompt_recall_passed']
  67e5ebfad8af26b1bb60e460: failed ['s2_attention_prompt_recall_passed']
  670e8b28f1a48a783a00c341: failed ['s1_attention_prompt_recall_passed']
  650b03136ab3d4c832d98b71: failed ['s1_attention_prompt_recall_passed']
  67722aa65639cba87301c81c: failed ['s2_attention_prompt_recall_passed']
  69811d9451120edfa666951d: failed ['s1_attention_prompt_rec

/var/folders/zz/78fqfyrj1lsdrh8ws2kmz9800000gn/T/ipykernel_66385/2127279233.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passed = df_participants[col].fillna(False).astype(bool).sum()
/var/folders/zz/78fqfyrj1lsdrh8ws2kmz9800000gn/T/ipykernel_66385/2127279233.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passed = df_participants[col].fillna(False).astype(bool).sum()
/var/folders/zz/78fqfyrj1lsdrh8ws2kmz9800000gn/T/ipykernel_66385/2127279233.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and w

## 5. Session Completion Checks

In [7]:
print("=" * 60)
print("SESSION COMPLETION CHECKS")
print("=" * 60)

for s in [1, 2]:
    prefix = f's{s}'
    print(f"\n--- Session {s} ---")

    # Has MBA data?
    has_mba = df_participants[f'{prefix}_mba_timestamp'].notna().sum()
    print(f"  Has MBA data: {has_mba}/{len(df_participants)}")

    # Has MBE data?
    has_mbe = df_participants[f'{prefix}_mbe_timestamp'].notna().sum()
    print(f"  Has MBE data: {has_mbe}/{len(df_participants)}")

    # Has chat messages?
    has_chat = (df_participants[f'{prefix}_num_total_messages'] > 0).sum()
    print(f"  Has chat messages: {has_chat}/{len(df_participants)}")

    # Message count stats
    user_msgs = df_participants[f'{prefix}_num_user_messages']
    print(f"  User messages — mean: {user_msgs.mean():.1f}, "
          f"median: {user_msgs.median():.0f}, "
          f"min: {user_msgs.min()}, max: {user_msgs.max()}")

    # Chat duration
    duration = df_participants[f'{prefix}_chat_duration']
    print(f"  Chat duration (sec) — mean: {duration.mean():.0f}, "
          f"values: {duration.value_counts().to_dict()}")

# Final survey
print(f"\n--- Final Survey ---")
print(f"  Has final survey: {df_participants['final_survey_timestamp'].notna().sum()}/{len(df_participants)}")

SESSION COMPLETION CHECKS

--- Session 1 ---
  Has MBA data: 228/228
  Has MBE data: 228/228
  Has chat messages: 228/228
  User messages — mean: 9.4, median: 9, min: 1, max: 35
  Chat duration (sec) — mean: 600, values: {600.0: 228}

--- Session 2 ---
  Has MBA data: 228/228
  Has MBE data: 228/228
  Has chat messages: 228/228
  User messages — mean: 10.7, median: 9, min: 1, max: 83
  Chat duration (sec) — mean: 600, values: {600.0: 228}

--- Final Survey ---
  Has final survey: 228/228


## 6. Conversation-Level Dataframe

In [8]:
def extract_conversation_messages(firebase_id, p_data):
    """Extract all conversation messages across both sessions."""
    messages = []

    url_params = p_data.get('urlParameters', {})
    prolific_id = url_params.get('PROLIFIC_PID', None)

    for session_num in [1, 2]:
        s_key = f'session{session_num}'
        session = p_data.get(s_key, {})

        # Messages are a list directly under session
        msg_data = session.get('messages', [])
        if isinstance(msg_data, list):
            msg_list = [m for m in msg_data if m is not None]
        elif isinstance(msg_data, dict):
            msg_list = [v for k, v in sorted(msg_data.items()) if v is not None]
        else:
            msg_list = []

        # Get system prompt for this session
        sys_prompt = session.get('systemPrompt', {}).get('prompt', None)

        for idx, msg in enumerate(msg_list):
            content = msg.get('content', '')
            message_row = {
                'participant_id': firebase_id,
                'prolific_id': prolific_id,
                'session': session_num,
                'conversation_id': f"{firebase_id}_session{session_num}",
                'message_index': idx,
                'message_id': msg.get('messageId', None),
                'role': msg.get('role', None),
                'content': content,
                'timestamp': msg.get('timestamp', None),
                'turn_number': msg.get('turnNumber', msg.get('messageId', None)),
                'system_prompt': msg.get('systemPrompt', sys_prompt),
                'message_length': len(content) if content else 0,
                'word_count': len(content.split()) if content else 0,
            }
            messages.append(message_row)

    return messages


# Build conversation dataframe
all_messages = []
for firebase_id, p_data in participant_data.items():
    all_messages.extend(extract_conversation_messages(firebase_id, p_data))

df_conversations = pd.DataFrame(all_messages)

print(f"Total messages before filtering: {len(df_conversations)}")

# Filter to only completed participants
valid_ids = set(df_participants['firebase_id'])
df_conversations = df_conversations[df_conversations['participant_id'].isin(valid_ids)].copy()

print(f"Total messages after filtering: {len(df_conversations)}")
print(f"Unique participants: {df_conversations['participant_id'].nunique()}")
print(f"\nMessages per session:")
print(df_conversations['session'].value_counts().sort_index())
print(f"\nMessages per role:")
print(df_conversations['role'].value_counts())

Total messages before filtering: 9616
Total messages after filtering: 9163
Unique participants: 228

Messages per session:
session
1    4288
2    4875
Name: count, dtype: int64

Messages per role:
role
user         4582
assistant    4581
Name: count, dtype: int64


## 7. Persona Vector Dataframe

In [9]:
# Bipolar trait poles for each trait dimension
TRAIT_POLES = {
    'empathy': ('empathetic', 'unempathetic'),
    'erudite': ('sophisticated', 'simplistic'),
    'robotic': ('robotic', 'human-like'),
    'romantic': ('romantic', 'platonic'),
    'sycophantic': ('sycophantic', 'honest'),
    'toxic': ('toxic', 'respectful'),
}


def extract_persona_vectors(firebase_id, p_data):
    """Extract persona vector history across both sessions from personaVectorLog."""
    rows = []

    url_params = p_data.get('urlParameters', {})
    prolific_id = url_params.get('PROLIFIC_PID', None)

    for session_num in [1, 2]:
        s_key = f'session{session_num}'
        session = p_data.get(s_key, {})
        persona_log = session.get('personaVectorLog', {})

        if not isinstance(persona_log, dict):
            continue

        for ts_key, entry in persona_log.items():
            if entry is None:
                continue

            persona_vector = entry.get('personaVector', {})
            row = {
                'firebase_id': firebase_id,
                'prolific_id': prolific_id,
                'session': session_num,
                'condition': entry.get('condition', None),
                'system_prompt': entry.get('systemPrompt', None),
                'timestamp': entry.get('timestamp', None),
                'timestamp_key': ts_key,
            }

            # Extract bipolar trait scores
            for trait, (pos_pole, neg_pole) in TRAIT_POLES.items():
                trait_data = persona_vector.get(trait, {})
                row[f'{trait}_{pos_pole}'] = trait_data.get(pos_pole, None)
                row[f'{trait}_{neg_pole}'] = trait_data.get(neg_pole, None)

            rows.append(row)

    return rows


# Build persona vector dataframe
all_vectors = []
for firebase_id, p_data in participant_data.items():
    all_vectors.extend(extract_persona_vectors(firebase_id, p_data))

df_persona_vectors = pd.DataFrame(all_vectors)

# Filter to completed participants
df_persona_vectors = df_persona_vectors[df_persona_vectors['firebase_id'].isin(valid_ids)].copy()

print(f"Persona Vector DataFrame shape: {df_persona_vectors.shape}")
print(f"Unique participants: {df_persona_vectors['firebase_id'].nunique()}")
print(f"\nVectors per session:")
print(df_persona_vectors['session'].value_counts().sort_index())
print(f"\nVectors per participant (mean): "
      f"{df_persona_vectors.groupby('firebase_id').size().mean():.1f}")
print(f"\nColumns: {list(df_persona_vectors.columns)}")

Persona Vector DataFrame shape: (5075, 19)
Unique participants: 228

Vectors per session:
session
1    2394
2    2681
Name: count, dtype: int64

Vectors per participant (mean): 22.3

Columns: ['firebase_id', 'prolific_id', 'session', 'condition', 'system_prompt', 'timestamp', 'timestamp_key', 'empathy_empathetic', 'empathy_unempathetic', 'erudite_sophisticated', 'erudite_simplistic', 'robotic_robotic', 'robotic_human-like', 'romantic_romantic', 'romantic_platonic', 'sycophantic_sycophantic', 'sycophantic_honest', 'toxic_toxic', 'toxic_respectful']


In [10]:
# Also build exploratory versions that include control participants
exploratory_ids = set(df_participants_exploratory['firebase_id'])

df_conversations_exploratory = pd.DataFrame(all_messages)
df_conversations_exploratory = df_conversations_exploratory[
    df_conversations_exploratory['participant_id'].isin(exploratory_ids)
].copy()

df_persona_vectors_exploratory = pd.DataFrame(all_vectors)
df_persona_vectors_exploratory = df_persona_vectors_exploratory[
    df_persona_vectors_exploratory['firebase_id'].isin(exploratory_ids)
].copy()

print("EXPLORATORY dataframes (includes control with data-write bug):")
print(f"  Conversations: {df_conversations_exploratory.shape} "
      f"({df_conversations_exploratory['participant_id'].nunique()} participants)")
print(f"  Persona vectors: {df_persona_vectors_exploratory.shape} "
      f"({df_persona_vectors_exploratory['firebase_id'].nunique()} participants)")
print(f"\nExploratory conversations by condition:")
merged = df_conversations_exploratory.merge(
    df_participants_exploratory[['firebase_id', 'condition_name']],
    left_on='participant_id', right_on='firebase_id', how='left'
)
print(merged.groupby('condition_name')['participant_id'].nunique())

EXPLORATORY dataframes (includes control with data-write bug):
  Conversations: (9461, 13) (239 participants)
  Persona vectors: (5246, 19) (239 participants)

Exploratory conversations by condition:
condition_name
control        73
multi_turn     81
single_turn    85
Name: participant_id, dtype: int64


## 8. Basic Statistics & Validation

In [11]:
print("=" * 60)
print("PARTICIPANT DATAFRAME SUMMARY")
print("=" * 60)

print(f"\nTotal completed participants: {len(df_participants)}")
print(f"Missing Prolific IDs: {df_participants['prolific_id'].isna().sum()}")
print(f"Missing session IDs: {df_participants['session_id'].isna().sum()}")

print(f"\n--- Pre-Survey Completion ---")
print(f"Has pre-survey: {df_participants['pre_survey_timestamp'].notna().sum()}/{len(df_participants)}")

print(f"\n--- MBA/MBE Trait Rating Ranges ---")
for s in [1, 2]:
    for phase in ['mba', 'mbe']:
        vals = []
        for trait in TRAITS:
            col = f's{s}_{phase}_{trait}'
            if col in df_participants.columns:
                vals.extend(df_participants[col].dropna().tolist())
        if vals:
            print(f"  S{s} {phase.upper()}: min={min(vals)}, max={max(vals)}, "
                  f"mean={np.mean(vals):.2f} (expected range: -10 to +10)")

PARTICIPANT DATAFRAME SUMMARY

Total completed participants: 228
Missing Prolific IDs: 0
Missing session IDs: 0

--- Pre-Survey Completion ---
Has pre-survey: 228/228

--- MBA/MBE Trait Rating Ranges ---
  S1 MBA: min=-10, max=10, mean=-0.92 (expected range: -10 to +10)
  S1 MBE: min=-10.0, max=10.0, mean=-1.30 (expected range: -10 to +10)
  S2 MBA: min=-10.0, max=10.0, mean=-0.89 (expected range: -10 to +10)
  S2 MBE: min=-10.0, max=10.0, mean=-1.80 (expected range: -10 to +10)


In [12]:
print("=" * 60)
print("CONVERSATION DATAFRAME SUMMARY")
print("=" * 60)

print(f"\nTotal messages: {len(df_conversations)}")
print(f"Unique participants: {df_conversations['participant_id'].nunique()}")

print(f"\nMessages per participant per session:")
msg_counts = df_conversations.groupby(['participant_id', 'session']).size().reset_index(name='count')
for s in [1, 2]:
    subset = msg_counts[msg_counts['session'] == s]['count']
    if len(subset) > 0:
        print(f"  Session {s}: mean={subset.mean():.1f}, median={subset.median():.0f}, "
              f"min={subset.min()}, max={subset.max()}")

print(f"\nAverage message length by role:")
print(df_conversations.groupby('role')[['message_length', 'word_count']].mean().round(1))

CONVERSATION DATAFRAME SUMMARY

Total messages: 9163
Unique participants: 228

Messages per participant per session:
  Session 1: mean=18.8, median=18, min=2, max=70
  Session 2: mean=21.4, median=18, min=2, max=166

Average message length by role:
           message_length  word_count
role                                 
assistant           868.6       135.6
user                 65.5        12.5


## 9. MBA vs MBE Comparison (Calibration Shift)

In [13]:
print("=" * 60)
print("MBA vs MBE TRAIT SHIFT (MBE - MBA)")
print("=" * 60)
print("Positive = participant rated trait higher after chat than before")

for s in [1, 2]:
    print(f"\n--- Session {s} ---")
    for trait in TRAITS:
        mba_col = f's{s}_mba_{trait}'
        mbe_col = f's{s}_mbe_{trait}'
        if mba_col in df_participants.columns and mbe_col in df_participants.columns:
            shift = df_participants[mbe_col] - df_participants[mba_col]
            valid = shift.dropna()
            if len(valid) > 0:
                print(f"  {trait:15s}: mean shift={valid.mean():+.2f}, "
                      f"std={valid.std():.2f}, n={len(valid)}")

MBA vs MBE TRAIT SHIFT (MBE - MBA)
Positive = participant rated trait higher after chat than before

--- Session 1 ---
  empathy        : mean shift=+2.44, std=6.34, n=228
  erudite        : mean shift=+1.11, std=5.34, n=228
  robotic        : mean shift=-0.37, std=6.71, n=228
  romantic       : mean shift=-0.91, std=5.25, n=228
  sycophantic    : mean shift=-2.05, std=6.06, n=228
  toxic          : mean shift=-2.54, std=5.87, n=228

--- Session 2 ---
  empathy        : mean shift=+0.64, std=7.29, n=228
  erudite        : mean shift=+0.30, std=6.50, n=228
  robotic        : mean shift=+0.28, std=6.24, n=228
  romantic       : mean shift=-2.06, std=4.63, n=228
  sycophantic    : mean shift=-2.31, std=5.95, n=228
  toxic          : mean shift=-2.29, std=5.24, n=228


df_participants.to_csv('data_participants.csv', index=False)
df_participants_exploratory.to_csv('data_participants_exploratory.csv', index=False)
df_conversations.to_csv('data_conversations.csv', index=False)
df_conversations_exploratory.to_csv('data_conversations_exploratory.csv', index=False)
df_persona_vectors.to_csv('data_persona_vectors.csv', index=False)
df_persona_vectors_exploratory.to_csv('data_persona_vectors_exploratory.csv', index=False)

print("Dataframes saved:")
print(f"  data_participants.csv:                {df_participants.shape}  (final analysis)")
print(f"  data_participants_exploratory.csv:     {df_participants_exploratory.shape}  (+ control)")
print(f"  data_conversations.csv:               {df_conversations.shape}")
print(f"  data_conversations_exploratory.csv:    {df_conversations_exploratory.shape}")
print(f"  data_persona_vectors.csv:             {df_persona_vectors.shape}")
print(f"  data_persona_vectors_exploratory.csv:  {df_persona_vectors_exploratory.shape}")

In [14]:
df_participants.to_csv('data_participants.csv', index=False)
df_participants_exploratory.to_csv('data_participants_exploratory.csv', index=False)
df_conversations.to_csv('data_conversations.csv', index=False)
df_persona_vectors.to_csv('data_persona_vectors.csv', index=False)

print("Dataframes saved:")
print(f"  data_participants.csv:             {df_participants.shape}  (final analysis only)")
print(f"  data_participants_exploratory.csv:  {df_participants_exploratory.shape}  (includes control w/ data-write bug)")
print(f"  data_conversations.csv:            {df_conversations.shape}")
print(f"  data_persona_vectors.csv:          {df_persona_vectors.shape}")

Dataframes saved:
  data_participants.csv:             (228, 97)  (final analysis only)
  data_participants_exploratory.csv:  (239, 97)  (includes control w/ data-write bug)
  data_conversations.csv:            (9163, 13)
  data_persona_vectors.csv:          (5075, 19)


## 11. Data Quality Preview

In [15]:
print("=" * 60)
print("MISSING DATA — PARTICIPANT DATAFRAME")
print("=" * 60)

missing = df_participants.isnull().sum()
missing_cols = missing[missing > 0].sort_values(ascending=False)

if len(missing_cols) > 0:
    print(f"\nColumns with missing values ({len(missing_cols)} columns):")
    for col, count in missing_cols.items():
        pct = count / len(df_participants) * 100
        print(f"  {col:50s}: {count:3d} ({pct:5.1f}%)")
else:
    print("\nNo missing values!")

print(f"\nNote: The following columns are EXPECTED to be null for condition 0 (control):")
print("  - final_ueq_* (UEQ only for conditions 1-2)")
print("  - final_viz_* (visualization questions only for conditions 1-2)")
print("  - final_drift_panel_helpful (condition 2 only)")
print("  - final_visualization_usage, final_visualization_feedback (conditions 1-2 only)")

MISSING DATA — PARTICIPANT DATAFRAME

Columns with missing values (20 columns):
  s1_prompt_shown                                   : 228 (100.0%)
  s2_prompt_shown                                   : 228 (100.0%)
  final_drift_panel_helpful                         : 148 ( 64.9%)
  final_ueq_complicated_easy                        :  63 ( 27.6%)
  final_visualization_usage                         :  63 ( 27.6%)
  final_ueq_usual_leading_edge                      :  63 ( 27.6%)
  final_ueq_conventional_inventive                  :  63 ( 27.6%)
  final_ueq_not_interesting_interesting             :  63 ( 27.6%)
  final_ueq_boring_exciting                         :  63 ( 27.6%)
  final_ueq_confusing_clear                         :  63 ( 27.6%)
  final_ueq_inefficient_efficient                   :  63 ( 27.6%)
  final_ueq_obstructive_supportive                  :  63 ( 27.6%)
  final_viz_comprehension                           :  63 ( 27.6%)
  final_viz_frequency                            

In [16]:
print("=" * 60)
print("MISSING DATA — CONVERSATION DATAFRAME")
print("=" * 60)

missing_conv = df_conversations.isnull().sum()
missing_conv_cols = missing_conv[missing_conv > 0]
if len(missing_conv_cols) > 0:
    print(f"\nColumns with missing values:")
    for col, count in missing_conv_cols.items():
        print(f"  {col}: {count}")
else:
    print("\nNo missing values!")

MISSING DATA — CONVERSATION DATAFRAME

No missing values!


## 12. Link Conversations to Participant Data

In [17]:
# Merge conversation data with participant conditions
df_combined = df_conversations.merge(
    df_participants[['firebase_id', 'condition_name', 'visualization_condition',
                     'prompt_order', 'prolific_id']],
    left_on='participant_id',
    right_on='firebase_id',
    how='left',
    suffixes=('', '_participant')
)

print(f"Combined dataframe shape: {df_combined.shape}")
print(f"\nSample:")
df_combined[['prolific_id', 'session', 'role', 'content', 'condition_name']].head(10)

Combined dataframe shape: (9163, 18)

Sample:


,prolific_id,session,role,content,condition_name
0,6637b21240220c2517fffa1a,1,user,Hi!,multi_turn
1,6637b21240220c2517fffa1a,1,assistant,Hello. How can I assist you today?,multi_turn
2,6637b21240220c2517fffa1a,1,user,I have 4 events coming up this spring and summ...,multi_turn
3,6637b21240220c2517fffa1a,1,assistant,Maintaining a positive atmosphere is crucial i...,multi_turn
4,6637b21240220c2517fffa1a,1,user,How do I greet them?,multi_turn
5,6637b21240220c2517fffa1a,1,assistant,"When greeting your ex and his wife, aim for a ...",multi_turn
6,6637b21240220c2517fffa1a,1,user,This is good advice except for calling them by...,multi_turn
7,6637b21240220c2517fffa1a,1,assistant,"In social settings, using first names can actu...",multi_turn
8,6637b21240220c2517fffa1a,1,user,I'm tempted to bring up politics because I wan...,multi_turn
9,6637b21240220c2517fffa1a,1,assistant,"It's understandable you'd be curious, but brin...",multi_turn


In [18]:
df_participants.head()

,firebase_id,prolific_id,session_id,study_id,visualization_condition,condition_name,assignment_method,prompt_order,session1_prompt_type,session2_prompt_type,...,final_ueq_confusing_clear,final_ueq_boring_exciting,final_ueq_not_interesting_interesting,final_ueq_conventional_inventive,final_ueq_usual_leading_edge,final_interaction_reflection,final_visualization_usage,final_visualization_feedback,final_general_feedback,attention_all_passed
0,1oHWwGGSCfRytaqXg6cD5GY6ZOq1,6637b21240220c2517fffa1a,69c95108ea060654e9b0a71f,69c85399187694d34f96b151,2,multi_turn,random,asst_first,ASST,ROLEPLY,...,4.0,6.0,7.0,7.0,7.0,Both interactions went very well. I preferred ...,I liked that it verified the impressions I was...,It looked more complicated than it was. I coul...,Worked very well. I would use this irl.,False
1,2AUwJGzTvqRj926cv32S2ZcH2eB3,672c7f537c4a19882d6f9e15,69c955206d2828d21dfdbd46,69c85399187694d34f96b151,2,multi_turn,random,asst_first,ASST,ROLEPLY,...,7.0,6.0,7.0,6.0,6.0,I did not found anything surprising and did no...,I analyzed data patterns and trends to inform ...,Visualizing patterns and trends made complex d...,No any feed back about interface or study in g...,False
3,5FXvkePciJNqG4cXzz4xjlbmZ5s1,69a6e19366f31b3e3460f2b7,69c956cedc6700d4f94ed94a,69c85399187694d34f96b151,1,single_turn,random,roleply_first,ROLEPLY,ASST,...,2.0,3.0,1.0,4.0,4.0,"The first one was far more fun and engaging, t...","I found it very convoluted and unhelpful, so I...","It looked pretty, and I like charts. I found i...",This was super fun and interesting! Thank you ...,False
4,5p1F8BaXN2apKUQmXLb4qh1l3Yx2,66f0409cf2658a66284b3b2d,69c955c575d08ec84b64ad9c,69c85399187694d34f96b151,1,single_turn,random,roleply_first,ROLEPLY,ASST,...,7.0,5.0,5.0,4.0,4.0,"It went well, the first bot was a lot more sar...",I checked it to see if the responses from ai s...,The visualization helped in seeing how the ai ...,no,False
5,8ATdWJSkcNP06Fcjg6q4hJ2ahp32,67e5ebfad8af26b1bb60e460,69c951132d15a40cca5c85d6,69c85399187694d34f96b151,0,control,random,roleply_first,ROLEPLY,ASST,...,NaN,NaN,NaN,NaN,NaN,"Nothing actually, the interaction was nice. al...",None,None,No,False
